In [1]:
%pip install pandas sqlalchemy groq matplotlib ipywidgets

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import io
import os
import ipywidgets as widgets
from IPython.display import display, clear_output
import pandas as pd
from sqlalchemy import create_engine, text
import matplotlib.pyplot as plt
from groq import Groq

# ==========================================
# 1. SETUP CLOUD ENGINE INSTANCES
# ==========================================

# Securely set Groq API credential parameter values
os.environ["GROQ_API_KEY"] = "API_KEY_HERE"  # Replace with your actual Groq API key
client = Groq()

# Global states to track runtime database contexts dynamically
engine = None
current_df = None
detected_schema = ""

# ==========================================
# 2. SCHEMA AND TRANSLATION AGENT LOGIC
# ==========================================

def extract_dataframe_schema(df: pd.DataFrame) -> str:
    """
    Analyzes any uploaded DataFrame and builds a concrete schema definition string
    to feed the LLM context pool.
    """
    schema_parts = ["Table: user_dataset", "Columns:"]
    for col, dtype in zip(df.columns, df.dtypes):
        # Convert pandas/numpy types to basic text-to-SQL types
        sql_type = "TEXT"
        if pd.api.types.is_numeric_dtype(dtype):
            sql_type = "REAL" if pd.api.types.is_float_dtype(dtype) else "INTEGER"
        elif pd.api.types.is_datetime64_any_dtype(dtype):
            sql_type = "DATETIME"
        schema_parts.append(f"  - {col} ({sql_type})")
    return "\n".join(schema_parts)

def generate_sql_query(user_question: str, schema_context: str) -> str:
    """
    Groq LLM Pipeline to parse natural intent directly down to single SELECT instructions.
    """
    system_instruction = f"""
    You are an expert Text-to-SQL compiler engine.
    Translate the user's plain English request into clean SQLite code targeting the 'user_dataset' table ONLY.
    
    Database Schema Layout:
    {schema_context}
    
    Rules for output:
    1. Output ONLY the raw executable SQL query string. Do not include markdown formatting, backticks, or text explanations.
    2. Always use explicit columns or expressions matching the given schema columns.
    3. Make sure all queries are highly efficient read-only SELECT statements.
    4. If column names contain empty spaces or special characters, wrap them in square brackets like [Column Name].
    """
    
    try:
        response = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[
                {"role": "system", "content": system_instruction},
                {"role": "user", "content": user_question}
            ],
            temperature=0.0 # Strict accuracy output constraint
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        return f"Error: LLM Processing error occurred: {str(e)}"

def validate_sql_security(sql_query: str) -> tuple[bool, str]:
    """
    Guardrail to ensure sandbox remains read-only.
    """
    upper_query = sql_query.upper()
    if "ERROR:" in upper_query:
        return False, sql_query
        
    destructive_keywords = ['DROP', 'DELETE', 'UPDATE', 'INSERT', 'ALTER', 'TRUNCATE', 'GRANT']
    for word in destructive_keywords:
        if word in upper_query:
            return False, f"Security Refusal: Destructive operational keyword '{word}' is strictly forbidden."
            
    if not upper_query.strip().startswith("SELECT"):
        return False, "Security Refusal: Executable syntax statement violates read-only SELECT requirements."
        
    return True, "Passed"

# ==========================================
# 3. INTERACTIVE CONTAINER DESIGN (UI)
# ==========================================

# Step A Layout Widgets: File Upload Form
file_uploader = widgets.FileUpload(
    accept='.csv',
    multiple=False,
    description='Upload CSV',
    button_style='info',
    icon='upload'
)
upload_output = widgets.Output()

# Step B Layout Widgets: Command Execution Dashboard
input_query = widgets.Text(
    value='Show me total sales revenue grouped by region',
    placeholder='Type your instructions or analysis questions here...',
    description='Command:',
    layout=widgets.Layout(width='75%'),
    disabled=True
)
run_button = widgets.Button(
    description='Process Query',
    button_style='success',
    icon='play',
    disabled=True
)
execution_output = widgets.Output()

# ==========================================
# 4. CONTROLLER INTERACTION EVENT LOOPS
# ==========================================

def on_file_uploaded(change):
    global engine, current_df, detected_schema
    with upload_output:
        clear_output()
        
        # Guard target files safely
        if not file_uploader.value:
            return
            
        try:
            # Grab the metadata description dictionary of uploaded file object
            uploaded_file = file_uploader.value[0]
            content = uploaded_file['content']
            
            # Read directly into virtual pandas frames
            current_df = pd.read_csv(io.BytesIO(content))
            
            # Build memory engine block instance
            engine = create_engine('sqlite:///:memory:')
            current_df.to_sql('user_dataset', engine, index=False, if_exists='replace')
            
            # Extract schemas dynamically
            detected_schema = extract_dataframe_schema(current_df)
            
            print(f"✅ Successfully Loaded File: '{uploaded_file['name']}' ({len(current_df)} rows discovered)")
            print("\n🔍 Automated Table Schema Detected:")
            print(detected_schema)
            
            # Unlock query widget sections immediately for the user
            input_query.disabled = False
            run_button.disabled = False
            
        except Exception as e:
            print(f"❌ Initialization failure: Couldn't read the file correctly. Error details: {e}")

def on_query_submitted(b):
    with execution_output:
        clear_output()
        
        if engine is None or current_df is None:
            print("❌ Setup error: Please upload a valid CSV source file above first.")
            return
            
        question = input_query.value.strip()
        if not question:
            print("⚠️ Prompt error: Please type an explicit data instruction message input.")
            return
            
        print(f"🧠 Step 1: Parsing user instruction logic: '{question}'...")
        
        # 1. Translate command to target SQL
        raw_sql = generate_sql_query(question, detected_schema)
        
        # Auto-clean common markdown backticks or bracket wrappers if returned by the LLM
        if raw_sql.startswith("```"):
            lines = raw_sql.splitlines()
            cleaned_lines = [line for line in lines if not line.startswith("```")]
            raw_sql = " ".join(cleaned_lines).strip()
        if raw_sql.startswith("sql"):
            raw_sql = raw_sql[3:].strip()
        if raw_sql.startswith("[") and raw_sql.endswith("]"):
            raw_sql = raw_sql[1:-1].strip()
            
        print(f"🤖 Step 2: Formulated Target Query Statement:\n{raw_sql}\n")
        
        # 2. Check Security Parameters
        is_safe, message = validate_sql_security(raw_sql)
        if not is_safe:
            print(f"❌ Guardrail Exception: {message}")
            return
            
        # 3. Process Execution Result Outcome
        print("📊 Step 3: Running inside dataset sandbox engine layer...")
        try:
            with engine.connect() as conn:
                result_df = pd.read_sql_query(text(raw_sql), conn)
            
            print("\n🎉 Execution Outcome Complete! Result Rows:")
            display(result_df)
            
            # 4. Optional Automatic Plotting Engine Context
            if not result_df.empty and len(result_df.columns) >= 2:
                numeric_cols = result_df.select_dtypes(include=['number']).columns
                object_cols = result_df.select_dtypes(exclude=['number']).columns
                
                if len(numeric_cols) > 0 and len(object_cols) > 0:
                    plt.figure(figsize=(7, 3.5))
                    plt.bar(result_df[object_cols[0]].astype(str), result_df[numeric_cols[0]], color='teal')
                    plt.title(f"Visualized: {numeric_cols[0]} relative to {object_cols[0]} values")
                    plt.ylabel(numeric_cols[0])
                    plt.xlabel(object_cols[0])
                    plt.xticks(rotation=30, ha='right')
                    plt.tight_layout()
                    plt.show()
                    
        except Exception as e:
            print(f"❌ Execution Failure: The query syntax generated broken database logic.")
            print(f"Detail Stacktrace Log: {str(e)}")

# Bind callbacks to component events
file_uploader.observe(on_file_uploaded, names='value')
run_button.on_click(on_query_submitted)

# ==========================================
# 5. INITIALIZE VIEW DISPLAY WINDOWS
# ==========================================
print("📁 STEP A: UPLOAD DATASET FOR PROCESSING")
display(file_uploader, upload_output)
print("\n💬 STEP B: CONVERSE NATURAL COMMAND STATEMENTS WITH DATA TARGETS")
display(widgets.HBox([input_query, run_button]), execution_output)

📁 STEP A: UPLOAD DATASET FOR PROCESSING


FileUpload(value=(), accept='.csv', button_style='info', description='Upload CSV')

Output()


💬 STEP B: CONVERSE NATURAL COMMAND STATEMENTS WITH DATA TARGETS


Output()